# 04 — Walk-forward Training + Backtest

Compara trend_following vs mean_reversion vs ML threshold. Métricas con fees+slippage.

In [ ]:
import polars as pl, yaml
from src.features import build_features
from src.strategies.trend_following import signal_trend_following
from src.strategies.mean_reversion import signal_mean_reversion
from src.backtesting.engine import backtest
from src.backtesting.metrics import summarize
from src.evaluation.statistics import compare_strategies
from src.evaluation.plots import plot_equity
df = build_features(pl.read_parquet("../data/processed/btcusdt_1m.parquet"))
# Estrategias rule-based
df_tf = signal_trend_following(df)
df_mr = signal_mean_reversion(df_tf)
bt_trend = backtest(df_tf, signal_col="signal_trend", fee_bps=10, slippage_bps=5, lag=1)
bt_mr = backtest(df_mr, signal_col="signal_mr", fee_bps=10, slippage_bps=5, lag=1)
print("Trend:", summarize(bt_trend))
print("MR:", summarize(bt_mr))
# ML walk-forward (requiere folds en configs/data.yaml)
# from src.models.training import train_predict
# preds, model = train_predict(df, feature_cols=["return_60m","volume_zscore","rsi_14","macd_hist"], folds=yaml.safe_load(open("../configs/data.yaml"))["validation"]["folds"])
compare_strategies({"trend": bt_trend, "mean_reversion": bt_mr})

In [ ]:
plot_equity(bt_trend, title="Trend Following — Equity")
plot_equity(bt_mr, title="Mean Reversion — Equity")